# Domain Classification - Difficulty Assignment

Assigns `difficulty` (Easy / Medium / Hard) to `bhashawiki_domain.jsonl`, a
500-row domain-classification set drawn from the Bhashik domain corpora.

Each row shows a passage of text and asks which of five academic domains it
belongs to:

| | |
|---|---|
| Task | 5-way classification, gold is the option **text**, not a letter |
| Options | Computer Science, Chemistry, Physics, Law, Mathematics |
| Metric | `accuracy` |
| Rows | 500, near-balanced: 110 / 109 / 99 / 93 / 89 |

**Option order is shuffled per row, and that is not optional here.** Every row
in this file offers the same five options in the same order, so the gold
position is decided entirely by the domain: Computer Science is always A,
Chemistry always B, and so on. A model with any preference for a particular
letter would therefore appear to be good at one domain and bad at another,
and the difficulty labels would record that preference rather than the
difficulty of the text. Shuffling with a per-row seed removes the effect at no
extra cost, since it is still one forward pass.

The shuffle is a scoring-time artifact only. The written output keeps the
original `options` order and the original `answer`, so the file stays
byte-identical to its source apart from `difficulty`.

**Protocol.** Three models classify each passage. The answer is read as an
argmax over the option-letter token ids in a single forward pass, so an
off-list answer is impossible and nothing has to be parsed. Votes sum:

| Correct | Difficulty |
|---|---|
| 3 / 3 | Easy |
| 2 / 3 | Medium |
| 0-1 / 3 | Hard |

**Baselines to read the result against.** Random guessing over five options
scores 20%; always answering the majority class scores 22%. A model near
either number is not classifying, and Cell 10 reports both alongside the
per-model accuracy.

**Runtime.** 500 rows x 3 models = 1500 single forward passes, roughly 15-25
minutes on a free T4. Progress is written every batch, so a disconnect resumes
where it stopped.

**Output.** `MCQ.jsonl` - the 14 schema fields with `difficulty` filled in -
plus `MCQ_audit.jsonl` recording every model's pick and the shuffle applied.

### Cell 1 - Install dependencies and authenticate

**An HF token is required.** Llama-3.1 and Gemma-2 are gated; only Mistral is
open. Accept each licence on huggingface.co, create a **read** token, then add
it in Colab via the **key icon** as a secret named `HF_TOKEN`.

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub

HF_OK = False
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
    HF_OK = True
    print("HF login OK")
except Exception as e:
    print("HF login skipped ({}). Gated models will fail to load.".format(e))

### Cell 2 - Mount Drive

Weights are cached to Drive as 4-bit copies, so the first run downloads and
quantises while every run after that loads straight from Drive.

In [ ]:
import os

DRIVE_OK    = False
DRIVE_MOUNT = "/drive"
CACHE_DIR   = os.path.join(DRIVE_MOUNT, "MyDrive", "models")

try:
    from google.colab import drive
    drive.mount(DRIVE_MOUNT, force_remount=True)
    DRIVE_OK = os.path.isdir(os.path.join(DRIVE_MOUNT, "MyDrive"))
except ImportError:
    print("Not running on Colab - Drive caching disabled.")
except Exception as e:
    print("DRIVE MOUNT FAILED: {}".format(e))

if DRIVE_OK:
    os.makedirs(CACHE_DIR, exist_ok=True)
    print("Drive mounted | weight cache: {}".format(CACHE_DIR))
else:
    print("\nWARNING: no Drive - weights will NOT be cached between sessions.")

### Cell 3 - Configuration

- `N_ROWS` - `None` scores the whole 500-row file, which is the default; the
  passages are short so there is little reason to sample.
- `SHUFFLE_OPTIONS` - leave this `True`. See the note at the top: with it off,
  the labels partly record which letter each model prefers.
- `SET_EVAL_METRIC` - written into every output row.

In [ ]:
import gc
import json
import random
import shutil
from collections import Counter, defaultdict

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---- paths ----
INPUT_FILE  = "bhashawiki_domain.jsonl"
OUTPUT_FILE = "MCQ.jsonl"
AUDIT_FILE  = "MCQ_audit.jsonl"
PROG_DIR    = "domain_progress"

# ---- sampling ----
N_ROWS = None            # None = the whole file
SEED   = 42

# ---- scoring ----
SHUFFLE_OPTIONS = True   # keep True: the source file has a fixed option order
BATCH_SIZE      = 25     # rows per save point

# ---- schema ----
SET_EVAL_METRIC = "accuracy"

# ---- models ----
MODELS = [
    {"name": "mistral", "repo": "mistralai/Mistral-7B-Instruct-v0.3"},   # ungated
    {"name": "llama",   "repo": "meta-llama/Llama-3.1-8B-Instruct"},     # GATED
    {"name": "gemma",   "repo": "google/gemma-2-9b-it", "attn": "eager"},# GATED
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,     # T4 has no bf16
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

SCHEMA_KEYS = [
    "id", "source", "category", "subcategory", "region", "language",
    "difficulty", "task_type", "question", "options", "answer",
    "explanation", "cultural_attr", "eval_metric",
]

TASK = ("Classify the academic domain of the passage below. The passage may be "
        "a transcript fragment and may read informally; judge it by its "
        "subject matter, not its style.")

os.makedirs(PROG_DIR, exist_ok=True)
print("input :", INPUT_FILE)
print("output:", OUTPUT_FILE)
print("shuffle options:", SHUFFLE_OPTIONS)
print("device:", "cuda" if torch.cuda.is_available() else "cpu")

### Cell 4 - Load, shuffle, and measure the baselines

The gold answer in this file is the option **text**, not a letter, so the gold
index is found by locating the answer among the options. A row whose answer
matches no option is dropped and reported rather than silently scored.

Each row then gets its own option order, permuted with a seed derived from the
row id. That makes the shuffle deterministic - re-running produces the same
permutation, so the audit file stays meaningful - while decorrelating the gold
position from the domain.

Finally, two baselines are printed. They are the numbers the per-model accuracy
in Cell 10 has to beat before any label here means anything.

In [ ]:
LETTERS = [chr(ord("A") + i) for i in range(26)]

with open(INPUT_FILE, encoding="utf-8") as f:
    all_rows = [json.loads(line) for line in f if line.strip()]
print("Loaded {} rows from {}".format(len(all_rows), INPUT_FILE))

usable, unresolved = [], []
for r in all_rows:
    opts = r.get("options") or []
    answer = str(r.get("answer") or "").strip()
    texts = [str(o).strip() for o in opts]
    if len(opts) < 2 or answer not in texts:
        unresolved.append(r)
        continue
    r["_gold_text"] = texts[texts.index(answer)]
    usable.append(r)

print("rows with a resolvable gold option: {}/{}".format(len(usable), len(all_rows)))
if unresolved:
    print("  DROPPED {} row(s) whose answer matches no option:".format(len(unresolved)))
    for r in unresolved[:3]:
        print("     [{}] answer={!r}".format(r["id"], r.get("answer")))

random.seed(SEED)
sample = usable if N_ROWS is None else random.sample(usable, min(N_ROWS, len(usable)))

# ---- per-row option permutation -------------------------------------------
for r in sample:
    opts = [str(o) for o in r["options"]]
    order = list(range(len(opts)))
    if SHUFFLE_OPTIONS:
        random.Random("{}|{}".format(SEED, r["id"])).shuffle(order)
    r["_order"] = order                                   # original index per slot
    r["_opts"] = [opts[i] for i in order]                 # what the model sees
    r["_gold"] = r["_opts"].index(r["_gold_text"])        # gold slot after shuffle

print("\nScoring {} rows".format(len(sample)))
print("  options per row: {}".format(dict(Counter(len(r["_opts"]) for r in sample))))

before = Counter(LETTERS[[str(o) for o in r["options"]].index(r["_gold_text"])] for r in sample)
after = Counter(LETTERS[r["_gold"]] for r in sample)
print("\n  gold position in the source file : {}".format(dict(sorted(before.items()))))
print("  gold position as presented       : {}".format(dict(sorted(after.items()))))
if SHUFFLE_OPTIONS:
    print("  -> the second row should be roughly flat; the first is fixed by domain")

# ---- baselines -------------------------------------------------------------
counts = Counter(r["_gold_text"] for r in sample)
majority = max(counts.values()) / len(sample)
rand = sum(1.0 / len(r["_opts"]) for r in sample) / len(sample)
print("\nbaselines to beat:")
print("  random guessing        {:.1%}".format(rand))
print("  always the top class   {:.1%}  ({})".format(majority, counts.most_common(1)[0][0]))
print("  class balance          {}".format(dict(counts.most_common())))

r = sample[0]
print("\n--- example row as the model will see it ---")
print("  {}".format(" ".join(str(r["question"]).split())[:150]))
for i, o in enumerate(r["_opts"]):
    print("     {}. {}{}".format(LETTERS[i], o, "   <- gold" if i == r["_gold"] else ""))

### Cell 5 - Build the prompt

The task line names the classification explicitly and warns that the passages
are transcript fragments, several of which read as spoken text rather than
formal prose. The prompt ends on the letter cue so that the very next token is
the one Cell 6 reads.

In [ ]:
def build_query(row):
    opts = "\n".join("{}. {}".format(LETTERS[i], o)
                     for i, o in enumerate(row["_opts"]))
    letters = "/".join(LETTERS[:len(row["_opts"])])
    return ("{}\n\n{}\n\nOptions:\n{}\n\nAnswer with one letter ({}):").format(
        TASK, " ".join(str(row["question"]).split()), opts, letters)


SYSTEM_PROMPT = (
    "You classify passages of text by academic domain.\n\n"
    "Reply with a SINGLE letter naming one of the given options. Output only "
    "that letter - no words, no punctuation, no explanation."
)


def build_completion(row):
    return SYSTEM_PROMPT + "\n\n" + build_query(row)


def build_chat_messages(row):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_query(row)}]


print("=" * 70)
print(build_completion(sample[0]))
print("=" * 70)
print("[gold: {}. {}]".format(LETTERS[sample[0]["_gold"]], sample[0]["_gold_text"]))

### Cell 6 - Constrained answering

One forward pass per row. The logits at the final position are compared only
across the token ids of the valid option letters, and the largest wins. The
model never generates free text, so there is nothing to parse, no truncation to
handle, and no way to answer off-list - which also means a refusal or a stray
"The answer is" cannot be scored as wrong by accident.

Both `A` and ` A` are checked, because most tokenizers treat the leading-space
form as a different token.

In [ ]:
def prompt_style(tokenizer):
    # base checkpoints have no chat template at all
    return "chat" if getattr(tokenizer, "chat_template", None) else "completion"


def letter_token_ids(tokenizer, n):
    ids = {}
    for let in LETTERS[:n]:
        variants = set()
        for form in (let, " " + let):
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc:
                variants.add(enc[0])
        ids[let] = sorted(variants)
    return ids


def format_prompt(tokenizer, row):
    if prompt_style(tokenizer) == "completion":
        return build_completion(row)
    msgs = build_chat_messages(row)
    try:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True)
    except Exception:
        # some templates (Gemma) reject a system role - fold it into the user
        # turn rather than dropping the instructions
        merged = [{"role": "user",
                   "content": msgs[0]["content"] + "\n\n" + msgs[1]["content"]}]
        return tokenizer.apply_chat_template(
            merged, tokenize=False, add_generation_prompt=True)


@torch.no_grad()
def choose(model, tokenizer, tok_cache, row):
    n = len(row["_opts"])
    if n not in tok_cache:
        tok_cache[n] = letter_token_ids(tokenizer, n)
    ids = tok_cache[n]

    text   = format_prompt(tokenizer, row)
    inputs = tokenizer(text, return_tensors="pt", truncation=True,
                       max_length=1024).to(model.device)
    logits = model(**inputs).logits[0, -1]

    best = max(LETTERS[:n],
               key=lambda let: max(logits[i].item() for i in ids[let]))
    return LETTERS.index(best)


def clear_hf_cache():
    shutil.rmtree("/root/.cache/huggingface/hub/", ignore_errors=True)
    gc.collect()
    torch.cuda.empty_cache()

print("Answering functions defined")

### Cell 7 - Load-or-cache, and the batched runner

A Drive copy is already 4-bit, so passing `quantization_config` again would
quantise twice - the branch below avoids that. `trust_remote_code=False`
throughout: all three are native architectures in `transformers`, and the flag
has caused loader failures elsewhere in this project.

`run_model` appends every batch to `domain_progress/<model>.jsonl` and reads it
back on start, so a Colab disconnect costs at most one batch and a model that
already finished is never loaded.

In [ ]:
def cache_path(spec):
    return os.path.join(CACHE_DIR, spec["name"] + "_4bit")


def load_model(spec):
    cached     = cache_path(spec)
    from_drive = DRIVE_OK and os.path.isfile(os.path.join(cached, "config.json"))
    source     = cached if from_drive else spec["repo"]

    kwargs = {"device_map": "auto", "trust_remote_code": False}
    if from_drive:
        how = "Drive cache (already 4-bit)"
    else:
        kwargs["quantization_config"] = bnb_config
        how = "HuggingFace download -> 4-bit"
    if spec.get("attn"):
        kwargs["attn_implementation"] = spec["attn"]
        how += ", attn=" + spec["attn"]

    print("  loading {} [{}]".format(source, how))
    tokenizer = AutoTokenizer.from_pretrained(source)
    model = AutoModelForCausalLM.from_pretrained(source, **kwargs).eval()

    if not from_drive and DRIVE_OK:
        print("  saving 4-bit copy to {} (one time)...".format(cached))
        os.makedirs(cached, exist_ok=True)
        model.save_pretrained(cached)
        tokenizer.save_pretrained(cached)
        print("  saved - future runs skip the download")

    print("  ready | VRAM: {:.2f}GB | prompt style: {}".format(
        torch.cuda.memory_allocated() / 1e9, prompt_style(tokenizer)))
    return model, tokenizer


def run_model(spec, rows):
    prog = os.path.join(PROG_DIR, spec["name"] + ".jsonl")

    done = {}
    if os.path.exists(prog):
        with open(prog, encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                done[item["id"]] = item
        print("  resuming - {}/{} already answered".format(len(done), len(rows)))

    remaining = [r for r in rows if r["id"] not in done]
    if not remaining:
        print("  {} already complete - skipping load".format(spec["name"]))
        return done

    model, tokenizer = load_model(spec)
    tok_cache = {}
    total = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

    for start in range(0, len(remaining), BATCH_SIZE):
        batch, results = remaining[start:start + BATCH_SIZE], []
        for row in batch:
            picked = choose(model, tokenizer, tok_cache, row)
            results.append({"id": row["id"], "picked": picked,
                            "gold": row["_gold"],
                            "picked_text": row["_opts"][picked],
                            "correct": int(picked == row["_gold"])})
        with open(prog, "a", encoding="utf-8") as f:
            for item in results:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")
        done.update({i["id"]: i for i in results})
        acc = sum(v["correct"] for v in done.values()) / len(done)
        print("  batch {}/{} saved - {}/{} rows | running acc {:.1%}".format(
            start // BATCH_SIZE + 1, total, len(done), len(rows), acc))

    del model, tokenizer
    clear_hf_cache()
    print("  {} complete".format(spec["name"]))
    return done

print("Runner defined")

### Cell 8 - Run all three models

Safe to re-run after a disconnect: finished rows are read back from
`domain_progress/` and a model that is already complete is skipped without
being loaded at all.

In [ ]:
answers = {}
for spec in MODELS:
    print("\n=== {} ===".format(spec["name"]))
    answers[spec["name"]] = run_model(spec, sample)

print("\nAll models done")

### Cell 9 - Assign difficulty and write the schema

Votes sum to a difficulty and the row is projected onto the 14 schema keys in
order. The working fields (`_opts`, `_order`, `_gold`, `_gold_text`) never
reach the output, so the written file keeps the **original** option order and
the original answer - the shuffle existed only for scoring.

In [ ]:
def get_difficulty(votes):
    score = sum(votes)
    if score == 3:
        return "Easy"
    if score == 2:
        return "Medium"
    return "Hard"


final_results, audit = [], []

for row in sample:
    votes = [answers[s["name"]][row["id"]]["correct"] for s in MODELS]
    difficulty = get_difficulty(votes)

    enriched = {**row, "difficulty": difficulty}
    if SET_EVAL_METRIC:
        enriched["eval_metric"] = SET_EVAL_METRIC
    final_results.append({k: enriched.get(k) for k in SCHEMA_KEYS})

    audit.append({
        "id":         row["id"],
        "difficulty": difficulty,
        "votes":      votes,
        "gold":       row["_gold_text"],
        "shown":      row["_opts"],
        "picks":      {s["name"]: answers[s["name"]][row["id"]]["picked_text"]
                       for s in MODELS},
        "question":   " ".join(str(row["question"]).split())[:300],
    })

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    for item in final_results:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
with open(AUDIT_FILE, "w", encoding="utf-8") as f:
    for item in audit:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Saved -> {} ({} rows)".format(OUTPUT_FILE, len(final_results)))
print("Audit -> {}".format(AUDIT_FILE))

### Cell 10 - Verify and report

Nothing here rebalances anything; it only makes the result legible.

- **Schema**: 14 keys in order, no null difficulty, and the question, options
  and answer of every row byte-identical to the input file - which is the check
  that the shuffle did not leak into the output.
- **Per-model accuracy against the two baselines.** A model at or below 22% is
  not classifying, and any label derived from it records noise.
- **Per-domain difficulty**, which is the interesting result: whether some
  subjects are systematically harder to identify than others.
- **Gold position vs difficulty.** After shuffling this should be flat. If one
  letter still stands out, the shuffle is not doing its job and the labels
  should not be trusted.

In [ ]:
bad_keys  = [r["id"] for r in final_results if list(r.keys()) != SCHEMA_KEYS]
null_diff = [r["id"] for r in final_results if not r["difficulty"]]
print("Schema check : {} rows | wrong keys: {} | null difficulty: {}".format(
    len(final_results), len(bad_keys), len(null_diff)))

src = {r["id"]: r for r in all_rows}
altered = [r["id"] for r in final_results
           if json.dumps([r["question"], r["options"], r["answer"]], ensure_ascii=False)
           != json.dumps([src[r["id"]]["question"], src[r["id"]]["options"],
                          src[r["id"]]["answer"]], ensure_ascii=False)]
print("question/options/answer altered: {}  (0 means the shuffle stayed internal)".format(
    len(altered)))

dist = Counter(r["difficulty"] for r in final_results)
print("\nDifficulty distribution:")
for lvl in ("Easy", "Medium", "Hard"):
    print("  {:<7}: {:4}  ({:.1%})".format(lvl, dist[lvl], dist[lvl] / len(final_results)))

print("\nPer model, against the baselines:")
print("  {:<10} {:>9}".format("", "accuracy"))
print("  {:<10} {:>8.1%}   random".format("(baseline)", rand))
print("  {:<10} {:>8.1%}   majority class".format("(baseline)", majority))
for s in MODELS:
    acc = sum(v["correct"] for v in answers[s["name"]].values()) / len(sample)
    flag = "   <- at or below the majority baseline" if acc <= majority else ""
    print("  {:<10} {:>8.1%}{}".format(s["name"], acc, flag))

print("\nPer domain:")
tab = defaultdict(Counter)
for a in audit:
    tab[a["gold"]][a["difficulty"]] += 1
print("  {:<18} {:>5} {:>6} {:>7} {:>6}   %Hard".format(
    "domain", "n", "Easy", "Medium", "Hard"))
for k in sorted(tab, key=lambda k: -tab[k]["Hard"] / max(sum(tab[k].values()), 1)):
    c = tab[k]; n = sum(c.values())
    print("  {:<18} {:>5} {:>6} {:>7} {:>6}   {:.0f}%".format(
        k, n, c["Easy"], c["Medium"], c["Hard"], 100 * c["Hard"] / n))

print("\nGold position vs difficulty (should be flat after shuffling):")
pos = defaultdict(Counter)
for row, a in zip(sample, audit):
    pos[LETTERS[row["_gold"]]][a["difficulty"]] += 1
rates = []
print("  gold {:>6} {:>7} {:>8}".format("n", "Easy", "%Easy"))
for let in sorted(pos):
    c = pos[let]; n = sum(c.values())
    rates.append(100 * c["Easy"] / n)
    print("   {}   {:>6} {:>7} {:>7.1f}%".format(let, n, c["Easy"], 100 * c["Easy"] / n))
print("  spread between the best and worst letter: {:.1f} points".format(
    max(rates) - min(rates)))

print("\nMost common confusions:")
conf = Counter()
for a in audit:
    for name, pick in a["picks"].items():
        if pick != a["gold"]:
            conf[(a["gold"], pick)] += 1
for (gold, pick), n in conf.most_common(6):
    print("  {:>16}  read as  {:<16} {:>4}".format(gold, pick, n))